# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test type per calendar month) from silver lab results and calendar, including a month-over-month LAG trend on abnormal rate, per the Silver-to-Gold STTM. Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

In [0]:
# Catalog placeholder from the STTM (<catalog>) is exposed as a widget
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog")

src = f"{catalog}.silver"
tgt_table = f"{catalog}.gold.lab_test_trends"
print(f"Source schema: {src} | Target table: {tgt_table}")

In [0]:
# Full rebuild: driving set = silver.lab_results grouped by (test_name, year, month)
# Calendar joined on the first day of the month (optional LEFT JOIN) for month_name / quarter_name.
spark.sql(f"""
CREATE OR REPLACE TABLE {tgt_table} AS
WITH monthly AS (
  SELECT test_name,
         year(test_date)                                         AS year,
         month(test_date)                                        AS month,
         COUNT(*)                                                AS total_tests,
         COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
         COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {src}.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT m.*,
         CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
         CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT r.*,
         LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  CAST(w.test_name AS STRING)                                  AS test_name,
  CAST(w.year AS INT)                                          AS year,
  CAST(w.month AS INT)                                         AS month,
  CAST(c.month_name AS STRING)                                 AS month_name,
  CAST(c.quarter_name AS STRING)                               AS quarter_name,
  CAST(w.total_tests AS INT)                                   AS total_tests,
  CAST(w.abnormal_test_count AS INT)                           AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                           AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))         AS prev_month_abnormal_rate_pct,
  CAST(CASE WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
            ELSE w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct
       END AS DECIMAL(6,2))                                    AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {src}.calendar c
  ON c.date_key = w.year * 10000 + w.month * 100 + 1
""")
print(f"{tgt_table} rebuilt.")

In [0]:
record_count = spark.table(tgt_table).count()
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
select * from adb_caresync.gold.lab_test_trends